# 04 - Baseline Match Model

Goal: turn team feature scores into match probabilities.

Important honesty note: this is a baseline model, not the final trained ML model yet. A trained model needs historical match results, such as `data/raw/results.csv`, where each row is a real international match with home team, away team, goals, and date.

Until that file is added, this notebook creates a strong first baseline from our engineered team features. This is still useful because it gives us match probabilities, contender rankings, and a structure we can later calibrate with real match outcomes.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

SRC_DIR = Path('../src').resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from match_modeling import (
    PROCESSED_DIR,
    RAW_DIR,
    load_team_features,
    build_pairwise_predictions,
    contender_table,
    predict_match,
    read_historical_results,
)

print('Notebook 04 imports loaded')

Notebook 04 imports loaded


## Step 1 - Load final team features

This file comes from Notebook 03. Each row is one national team, and the important modeling columns are:

- `power_score`: overall squad strength
- `attack_score`: goal threat
- `creation_score`: chance creation
- `defense_score`: defensive actions / resistance
- `keeper_score`: goalkeeper signal
- `depth_score`: strength of the selected squad
- `data_confidence_score`: how much we trust the data behind the team row

A model cannot use player rows directly for match prediction. It needs Team A features and Team B features.

In [2]:
teams = load_team_features()
display(teams.head())

print(f'Teams available for baseline modeling: {len(teams)}')

Loaded team features: 45 teams x 26 columns
['nation_code', 'nation_name', 'selection_method', 'squad_players_used', 'estimated_players_used', 'verified_supplemental_used', 'avg_selection_score', 'top6_goal_form_sum', 'top6_goal_form_mean', 'top10_chance_form_sum', 'top10_chance_form_mean', 'attacker_goal_form_mean', 'midfielder_chance_form_mean', 'defender_actions_per90_mean', 'keeper_save_pct_best', 'squad_weighted_minutes_mean', 'attack_score', 'creation_score', 'defense_score', 'keeper_score', 'depth_score', 'data_confidence_score', 'coverage_pct', 'has_announced_squad_file', 'raw_power_score', 'power_score']


,nation_code,nation_name,selection_method,squad_players_used,estimated_players_used,verified_supplemental_used,avg_selection_score,top6_goal_form_sum,top6_goal_form_mean,top10_chance_form_sum,...,attack_score,creation_score,defense_score,keeper_score,depth_score,data_confidence_score,coverage_pct,has_announced_squad_file,raw_power_score,power_score
0,ENG,England,official_squad,26,1,0,74.863565,3.281209,0.546868,1.454975,...,95.000000,96.777778,78.444444,86.666667,88.888889,98.653846,1.0,True,90.853333,90.547577
1,GER,Germany,official_squad,25,1,0,76.807714,2.591882,0.431980,1.338744,...,92.444444,94.444444,71.555556,84.444444,93.333333,98.600000,1.0,True,88.313333,88.004237
2,FRA,France,official_squad,26,1,0,77.163462,3.649274,0.608212,1.276214,...,96.555556,88.888889,45.555556,93.333333,95.555556,98.653846,1.0,True,84.972222,84.686258
3,ESP,Spain,official_squad,26,2,0,75.382369,2.518759,0.419793,0.963443,...,81.888889,74.555556,91.555556,95.555556,91.111111,97.307692,1.0,True,84.357778,83.789985
4,ARG,Argentina,official_squad_capped_top26,26,4,1,79.975258,2.476049,0.412675,1.330950,...,85.222222,88.000000,91.333333,18.888889,97.777778,94.615385,1.0,True,80.312222,79.231096


Teams available for baseline modeling: 45


## Step 2 - Check the contender table

This is not yet a World Cup winner prediction. It is simply the current squad-strength ranking from our engineered features.

This is where your France question matters: France can have the best attack while England has the best overall score. Overall score combines attack with creation, defense, keeper, depth, and data confidence.

In [3]:
contenders = contender_table(teams, top_n=20)
display(contenders)

print('Attack ranking:')
display(
    teams[['nation_code', 'nation_name', 'attack_score', 'power_score']]
    .sort_values('attack_score', ascending=False)
    .head(12)
)

,nation_code,nation_name,power_score,attack_score,creation_score,defense_score,keeper_score,depth_score,data_confidence_score,selection_method
0,ENG,England,90.547577,95.000000,96.777778,78.444444,86.666667,88.888889,98.653846,official_squad
1,GER,Germany,88.004237,92.444444,94.444444,71.555556,84.444444,93.333333,98.600000,official_squad
2,FRA,France,84.686258,96.555556,88.888889,45.555556,93.333333,95.555556,98.653846,official_squad
3,ESP,Spain,83.789985,81.888889,74.555556,91.555556,95.555556,91.111111,97.307692,official_squad
4,ARG,Argentina,79.231096,85.222222,88.000000,91.333333,18.888889,97.777778,94.615385,official_squad_capped_top26
5,SEN,Senegal,78.586558,82.333333,73.555556,84.000000,77.777778,80.000000,94.615385,official_squad_capped_top26
6,NED,Netherlands,78.394444,93.666667,95.777778,52.222222,18.888889,100.000000,100.000000,probable_top26
7,BEL,Belgium,78.071589,90.666667,77.222222,52.666667,88.888889,82.222222,93.269231,official_squad
8,POR,Portugal,77.614269,88.222222,95.555556,46.888889,56.666667,86.666667,94.615385,official_squad
9,BRA,Brazil,70.159231,85.111111,84.666667,28.666667,56.666667,84.444444,93.269231,official_squad


Attack ranking:


,nation_code,nation_name,attack_score,power_score
2,FRA,France,96.555556,84.686258
0,ENG,England,95.000000,90.547577
6,NED,Netherlands,93.666667,78.394444
1,GER,Germany,92.444444,88.004237
7,BEL,Belgium,90.666667,78.071589
8,POR,Portugal,88.222222,77.614269
4,ARG,Argentina,85.222222,79.231096
9,BRA,Brazil,85.111111,70.159231
5,SEN,Senegal,82.333333,78.586558
3,ESP,Spain,81.888889,83.789985


## Step 3 - Build baseline match probabilities

The baseline model compares Team A and Team B.

It creates two ideas:

- attacking pressure: Team A attack/creation/depth against Team B defense/keeper/depth
- overall edge: Team A `power_score` minus Team B `power_score`

Then it turns the edge into three probabilities:

- Team A win
- Draw
- Team B win

Why include draw? Group-stage football has draws. Knockout football does not, but later simulation can convert draw-like outcomes into extra time / penalties.

In [4]:
pairwise = build_pairwise_predictions(teams)
print('Pairwise predictions:', pairwise.shape)
display(pairwise.head(10))

Pairwise predictions: (990, 8)


,team_a,team_b,team_a_code,team_b_code,team_a_win_prob,draw_prob,team_b_win_prob,net_edge
43,England,Canada,ENG,CAN,0.919898,0.08,0.000102,109.332770
86,Germany,Canada,GER,CAN,0.919863,0.08,0.000137,105.772807
169,Spain,Canada,ESP,CAN,0.919840,0.08,0.000160,103.855865
42,England,Ecuador,ENG,ECU,0.919780,0.08,0.000220,100.042909
128,France,Canada,FRA,CAN,0.919779,0.08,0.000221,100.017146
41,England,Australia,ENG,AUS,0.919776,0.08,0.000224,99.820964
85,Germany,Ecuador,GER,ECU,0.919704,0.08,0.000296,96.482946
84,Germany,Australia,GER,AUS,0.919698,0.08,0.000302,96.261001
248,Senegal,Canada,SEN,CAN,0.919663,0.08,0.000337,94.956094
168,Spain,Ecuador,ESP,ECU,0.919652,0.08,0.000348,94.566004


## Step 4 - Query specific matchups

This helper lets us inspect any two teams. Try changing the country codes.

In [5]:
def show_match(team_a_code, team_b_code):
    team_a = teams[teams['nation_code'] == team_a_code].iloc[0]
    team_b = teams[teams['nation_code'] == team_b_code].iloc[0]
    pred = predict_match(team_a, team_b)
    return pd.DataFrame([pred])

display(show_match('ENG', 'FRA'))
display(show_match('FRA', 'BRA'))
display(show_match('ARG', 'GER'))

,team_a,team_b,team_a_code,team_b_code,team_a_win_prob,draw_prob,team_b_win_prob,net_edge
0,England,France,ENG,FRA,0.509253,0.256438,0.234309,9.315624


,team_a,team_b,team_a_code,team_b_code,team_a_win_prob,draw_prob,team_b_win_prob,net_edge
0,France,Brazil,FRA,BRA,0.718848,0.168935,0.112216,22.286656


,team_a,team_b,team_a_code,team_b_code,team_a_win_prob,draw_prob,team_b_win_prob,net_edge
0,Argentina,Germany,ARG,GER,0.203496,0.23195,0.564554,-12.244706


## Step 5 - Save outputs

We save:

- `baseline_pairwise_match_predictions.csv`: every team-vs-team probability
- `contender_power_ranking.csv`: top teams by engineered squad strength

These are useful for debugging and for a future frontend.

In [6]:
pairwise_path = PROCESSED_DIR / 'baseline_pairwise_match_predictions.csv'
contenders_path = PROCESSED_DIR / 'contender_power_ranking.csv'

pairwise.to_csv(pairwise_path, index=False)
contenders.to_csv(contenders_path, index=False)

print('Saved:', pairwise_path)
print('Saved:', contenders_path)

Saved: C:\Users\sambi\OneDrive\Desktop\worldcup-predictor\data\processed\baseline_pairwise_match_predictions.csv
Saved: C:\Users\sambi\OneDrive\Desktop\worldcup-predictor\data\processed\contender_power_ranking.csv


## Step 6 - Historical results check

For actual trained ML, we need historical international match results. The expected file path is:

`data/raw/results.csv`

A good dataset usually has columns like:

- `date`
- `home_team`
- `away_team`
- `home_score`
- `away_score`
- `tournament`

Once that file exists, Notebook 05 can train a real model using Team A vs Team B feature differences as inputs and match result as the target.

In [7]:
historical_results = read_historical_results(RAW_DIR / 'results.csv')

if historical_results is None:
    print('Next required file for trained ML: data/raw/results.csv')
else:
    display(historical_results.head())

Historical results not found at C:\Users\sambi\OneDrive\Desktop\worldcup-predictor\data\raw\results.csv
Next required file for trained ML: data/raw/results.csv
